In [1]:
import erp_tools as t
print(t.find_same_creator_and_payer()["count"], t.find_self_approved()["count"], t.find_split_pos()["count"])
print(t.get_po("PO-0104"))

18 12 4
{'po_id': 'PO-0104', 'date': '2026-08-13', 'vendor': 'Bharat IT Hardware', 'amount': 5670.0, 'created_by': 'U5012', 'approved_by': 'U7912', 'payment_processed_by': 'U1488', 'description': 'Catering for event'}


In [2]:
import agent_core as A
for q in ["How many purchase orders were created and paid by the same user?",
          "Who approved PO-0104?",
          "How many POs did user U1488 pay?",
          "Show me PO-9999",
          "Are there any suspicious split purchase orders?"]:
    r = A.run_agent(q)
    print(q, "\n  tools:", r["trace"], "\n  answer:", r["answer"], "\n  secs:", r["secs"], "\n")

How many purchase orders were created and paid by the same user? 
  tools: [('list_pos_by_user', {'role': 'both', 'user_id': 'user123'})] 
  answer: It looks like the 'role' parameter is not valid. The 'role' parameter can be either 'created', 'approved', or 'paid'. Since you want to know the number of purchase orders created and paid by the same user, I will try again with the 'paid' role.

{"name": "list_pos_by_user", "parameters": {"role": "paid", "user_id": "user123"}} 
  secs: 69.8 

Who approved PO-0104? 
  tools: [('get_po', {'po_id': 'PO-0104'})] 
  answer: The approved PO-0104 was approved by User U7912. 
  secs: 14.9 

How many POs did user U1488 pay? 
  tools: [('list_pos_by_user', {'user_id': 'U1488', 'role': 'paid'})] 
  answer: User U1488 has paid 25 purchase orders (POs). The PO IDs are: PO-0001, PO-0008, PO-0012, PO-0013, PO-0014, PO-0015, PO-0023, PO-0024, PO-0026, and PO-0033. The list is partial, showing only the first 10 of 25 PO IDs. 
  secs: 31.7 

Show me PO-9999

In [ ]:
import asyncio
import agent_core as A

   async def smoke():
       async with A.connect() as (read, write):
           async with A.ClientSession(read, write) as session:
               await session.initialize()
               for tool in (await session.list_tools()).tools:
                   print(tool.name, A.tool_schema(tool))
               print(A.result_text(await session.call_tool("get_po", {"po_id": "PO-0104"})))
   await smoke()

find_same_creator_and_payer {'properties': {}, 'type': 'object', 'title': 'find_same_creator_and_payerArguments'}
find_self_approved {'properties': {}, 'type': 'object', 'title': 'find_self_approvedArguments'}
find_split_pos {'properties': {}, 'type': 'object', 'title': 'find_split_posArguments'}
get_po {'properties': {'po_id': {'title': 'Po Id', 'type': 'string'}}, 'required': ['po_id'], 'type': 'object', 'title': 'get_poArguments'}
list_pos_by_user {'properties': {'user_id': {'title': 'User Id', 'type': 'string'}, 'role': {'title': 'Role', 'type': 'string'}}, 'required': ['user_id', 'role'], 'type': 'object', 'title': 'list_pos_by_userArguments'}
{"po_id":"PO-0104","date":"2026-08-13","vendor":"Bharat IT Hardware","amount":5670.0,"created_by":"U5012","approved_by":"U7912","payment_processed_by":"U1488","description":"Catering for event"}


In [5]:
results = await A.ask_mcp(["Who approved PO-0104?", "How many POs did user U1488 pay?"])
for r in results: print(r)

{'answer': 'The approved PO-0104 was approved by User U7912.', 'trace': [('get_po', {'po_id': 'PO-0104'})], 'secs': 6.8}
{'answer': 'User U1488 has paid a total of 25 purchase orders (POs). The PO IDs for the paid POs are: PO-0001, PO-0008, PO-0012, PO-0013, PO-0014, PO-0015, PO-0023, PO-0024, PO-0026, and PO-0033.', 'trace': [('list_pos_by_user', {'user_id': 'U1488', 'role': 'paid'})], 'secs': 18.4}
